# CausalMan: RCA data generation

This notebook generates the interventional datasets for the CausalMan root-cause analysis (RCA) benchmark.
The RCA task asks: given samples drawn from a system under an unknown intervention, identify which
variable(s) were intervened upon and characterise the intervention.

There are four tasks, each run on two scales, giving eight datasets in total:

| Task | Scales | Scenario |
|---|---|---|
| s01 | micro, small | Hard (atomic) intervention: press-fitting force locked to 17 000 N |
| s02 | micro, small | Soft intervention: max allowable force drawn from Normal(18 500, 3 000) |
| s03 | medium, large | Soft intervention on both T1 and T2 max force simultaneously |
| s04 | medium, large | Mixed: two observed soft interventions + one hidden hard intervention on `MV1_Emv` |

Each dataset folder contains:

| File | Description |
|---|---|
| `causalman_<scale>_do(...).csv` | Observable node samples under the named intervention |
| `intervention_mask.csv` | Boolean per-row mask — `True` where the intervention was applied |
| `interventions.json` | Machine-readable intervention spec (variable, kind, parameters) |

**Edit the configuration cell below, then Run All.**

In [ ]:
# ── The only cell you need to edit ────────────────────────────────────────────

# Choose which tasks and scales to generate.
# Tasks are automatically skipped for scales outside their compatible set
# (s01/s02 → micro/small; s03/s04 → medium/large).
TASK_IDS    = ["s02"]        # any subset of ["s01", "s02", "s03", "s04"]
SCALES      = ["micro", "small"]      # any subset of ["micro", "small", "medium", "large"]
SEED        = 42
N_SAMPLES   = 10_000
OUTPUT_ROOT = "output/causalman_rca"

# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import json
import os
from pathlib import Path
import sys

from sympy.stats import Normal

# Put the repository root before this notebook directory so causalman.py
# cannot shadow the causalman package when the notebook runs in-place.
PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / "pyproject.toml").is_file()
                     and (path / "causalman" / "__init__.py").is_file()), None)
if PROJECT_ROOT is not None:
    project_root = str(PROJECT_ROOT)
    if project_root in sys.path:
        sys.path.remove(project_root)
    sys.path.insert(0, project_root)

from causalman import CausalMan

# ── Task registry ─────────────────────────────────────────────────────────────
# Each entry defines one RCA scenario. "scales" lists the CausalMan variants
# compatible with that task; tasks are skipped for scales not in SCALES.
#
# kind="constant" → hard/atomic intervention  do(X = value)
# kind="normal"   → soft/stochastic intervention  do(X ~ Normal(mean, std))
TASKS = [
    {
        # s01: T1 press-fitting force fixed to an abnormally high constant.
        # Hard intervention — the variable is fully determined, no noise.
        "task_id": "s01",
        "slug":    "force_17000",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Force", "kind": "constant", "value": 17000.0},
        ],
    },
    {
        # s02: Max allowable press-fitting force shifted to a higher distribution.
        # Soft intervention — the variable retains its stochasticity.
        "task_id": "s02",
        "slug":    "fmax_normal",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
        ],
    },
    {
        # s03: Both T1 and T2 max forces shifted simultaneously — multi-target soft intervention.
        "task_id": "s03",
        "slug":    "two_fmax_normal",
        "scales":  ["medium", "large"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
            {"variable": "PF_M1_T2_Fmax", "kind": "normal", "mean": 19500.0, "std": 4000.0},
        ],
    },
    {
        # s04: Mixed scenario — two observed soft interventions plus one hidden hard
        # intervention on MV1_Emv (a latent node absent from the observed CSV).
        # Tests whether methods can detect a root cause they cannot directly observe.
        "task_id": "s04",
        "slug":    "mixed_with_hidden_emv",
        "scales":  ["medium", "large"],
        "interventions": [
            {"variable": "MV2_DmvMax",     "kind": "normal",   "mean": 4.7,     "std": 1.0},
            {"variable": "PF_M1_T1_Force", "kind": "normal",   "mean": 16500.0, "std": 3000.0},
            # MV1_Emv is latent — it does not appear in the observed CSV.
            {"variable": "MV1_Emv",        "kind": "constant", "value": 190000.0},
        ],
    },
]

In [3]:
from datetime import datetime

now = datetime.now().strftime("%Y_%m_%d_%H%M%S")
OUTPUT_ROOT = f"{OUTPUT_ROOT}_{now}"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Populated lazily on first encounter of each scale. Sharing them across tasks
# guarantees every dataset for the same scale has an identical column list.
observable_by_scale = {}
dag_by_scale = {}

for task in [t for t in TASKS if t["task_id"] in TASK_IDS]:
    for scale in [s for s in task["scales"] if s in SCALES]:
        dataset_id = f"rca_{task['task_id']}_{scale}_{task['slug']}"
        out_dir = os.path.join(OUTPUT_ROOT, dataset_id)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n── {dataset_id} ──")

        # ── Observable column list ────────────────────────────────────────────
        # Run one clean (no-intervention) simulation to discover which columns
        # the DAG marks as observable and are non-constant. Two filters apply:
        #   (a) DAG attribute Observable = True or "Observable"
        #   (b) column has more than one unique value in unperturbed data
        if scale not in observable_by_scale:
            sim = CausalMan(
                name=f"causalman_{scale}", seed=SEED, batch_multiplier=1,
                parallelize=True, save_path=os.path.join(OUTPUT_ROOT, f"_obs_{scale}"),
            )
            df_obs, _, _, dag = sim.sample()
            dag_observable = [
                n for n, d in dag.nodes(data=True)
                if d.get("Observable") in (True, "Observable") and n in df_obs.columns
            ]
            observable = [col for col in dag_observable if df_obs[col].nunique(dropna=False) > 1]
            observable_by_scale[scale] = observable
            dag_by_scale[scale] = dag
            print(f"  {scale}: {len(observable)} observable columns  "
                  f"({len(dag_observable) - len(observable)} constant columns removed)")

        observable = observable_by_scale[scale]

        # ── Interventional sample ─────────────────────────────────────────────
        # Build the dict the simulator expects: plain float for hard interventions,
        # sympy Normal for soft ones.
        intervention_dict = {}
        for iv in task["interventions"]:
            if iv["kind"] == "constant":
                intervention_dict[iv["variable"]] = iv["value"]
            else:
                intervention_dict[iv["variable"]] = Normal(iv["variable"], iv["mean"], iv["std"])

        sim = CausalMan(
            name=f"causalman_{scale}", seed=SEED, batch_multiplier=1,
            parallelize=True, save_path=os.path.join(out_dir, "_simulator"),
        )
        sim.intervention_dict = intervention_dict
        df, intervention_mask, _, _ = sim.sample()

        if len(df) < N_SAMPLES:
            raise RuntimeError(
                f"Simulator produced {len(df)} rows but {N_SAMPLES} are required. "
                "Increase batch_multiplier or reduce N_SAMPLES."
            )

        # ── Write outputs ─────────────────────────────────────────────────────
        # The CSV filename encodes the intervention in do-calculus notation so
        # the dataset is self-describing without opening interventions.json.
        do_parts = []
        for iv in task["interventions"]:
            if iv["kind"] == "constant":
                do_parts.append(f"{iv['variable']}={iv['value']}")
            else:
                do_parts.append(f"{iv['variable']}=Normal({iv['mean']},{iv['std']})")
        do_str = "do(" + ",".join(do_parts) + ")"

        # Sample the same row indices for all three files so they stay aligned.
        idx = df.sample(n=N_SAMPLES, random_state=SEED).index
        df.loc[idx, observable].to_csv(os.path.join(out_dir, f"causalman_{scale}_{do_str}.csv"), index=False)
        # intervention_mask has one boolean column per intervened variable;
        # True for rows where that variable's intervention was active.
        intervention_mask.loc[idx].to_csv(os.path.join(out_dir, "intervention_mask.csv"), index=False)

        with open(os.path.join(out_dir, "interventions.json"), "w") as f:
            json.dump(task["interventions"], f, indent=2)

        targets = [iv["variable"] for iv in task["interventions"]]
        print(f"  {N_SAMPLES:,} rows  |  {len(observable)} observable columns")
        print(f"  intervention targets: {targets}")

print(f"\nDone → {os.path.abspath(OUTPUT_ROOT)}")


── rca_s01_micro_force_17000 ──
Starting simulation for production line 0 out of 1
Finished sampling
  micro: 24 observable columns  (29 constant columns removed)
Starting simulation for production line 0 out of 1
Finished sampling
  10,000 rows  |  24 observable columns
  intervention targets: ['PF_M1_T1_Force']

── rca_s01_small_force_17000 ──
Starting simulation for production line 0 out of 2
Starting simulation for production line 1 out of 2
Finished sampling
  small: 52 observable columns  (1 constant columns removed)
Starting simulation for production line 0 out of 2
Starting simulation for production line 1 out of 2
Finished sampling
  10,000 rows  |  52 observable columns
  intervention targets: ['PF_M1_T1_Force']

Done → c:\Users\tan2rng\CausalMan\src\output\causalman_rca_2026_07_21_174240


In [4]:
from graph_projections import get_latent_projection_single as latent_projection
from graph_projections import count_edge_types, write_mixed_graph_graphml, admg2mag, validate_mag

# ── Ground-truth causal graphs ────────────────────────────────────────────────
# Project the full DAG (which includes latent nodes) onto the observable nodes.
# Latent common causes become bidirected edges in the projected graph.
#
# Two representations are saved per scale:
#   ADMG (Acyclic Directed Mixed Graph) — direct output of the latent projection
#   MAG  (Maximal Ancestral Graph)       — canonical form used by many CI algorithms
for scale in observable_by_scale:
    dag = dag_by_scale[scale]
    observable = observable_by_scale[scale]
    observable_set = set(observable)

    projection_dag = dag.copy()
    for node in projection_dag.nodes():
        projection_dag.nodes[node]["Observable"] = node in observable_set

    n_latent = projection_dag.number_of_nodes() - len(observable_set)
    print(f"\ncausalman_{scale}: {len(observable_set)} observable nodes | {n_latent} latent nodes")

    try:
        projected_admg = latent_projection(projection_dag)
        directed_count, bidirected_count = count_edge_types(projected_admg)
        print(f"  ADMG: {projected_admg.number_of_nodes()} nodes | {directed_count} directed edges | {bidirected_count} bidirected edges")
        admg_path = os.path.join(OUTPUT_ROOT, f"causalman_{scale}_ground_truth_admg.graphml")
        write_mixed_graph_graphml(projected_admg, admg_path)
        print(f"  Saved ADMG → {admg_path}")
    except Exception as e:
        projected_admg = None
        print(f"  ADMG creation failed: {e}")

    try:
        mag = admg2mag(projected_admg)
        validate_mag(mag)
        directed_count_mag, bidirected_count_mag = count_edge_types(mag)
        print(f"  MAG:  {mag.number_of_nodes()} nodes | {directed_count_mag} directed edges | {bidirected_count_mag} bidirected edges")
        mag_path = os.path.join(OUTPUT_ROOT, f"causalman_{scale}_ground_truth_mag.graphml")
        write_mixed_graph_graphml(mag, mag_path)
        print(f"  Saved MAG  → {mag_path}")
    except Exception as e:
        mag = None
        print(f"  MAG creation failed: {e}")


causalman_micro: 24 observable nodes | 133 latent nodes
  ADMG: 24 nodes | 40 directed edges | 127 bidirected edges
  Saved ADMG → output/causalman_rca_2026_07_21_174240\causalman_micro_ground_truth_admg.graphml
  MAG:  24 nodes | 90 directed edges | 96 bidirected edges
  Saved MAG  → output/causalman_rca_2026_07_21_174240\causalman_micro_ground_truth_mag.graphml

causalman_small: 52 observable nodes | 105 latent nodes
  ADMG: 52 nodes | 98 directed edges | 13 bidirected edges
  Saved ADMG → output/causalman_rca_2026_07_21_174240\causalman_small_ground_truth_admg.graphml
  MAG:  52 nodes | 110 directed edges | 11 bidirected edges
  Saved MAG  → output/causalman_rca_2026_07_21_174240\causalman_small_ground_truth_mag.graphml
